In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict, Literal
import os
from typing import TypedDict, Annotated
import operator

In [3]:
load_dotenv()

if not os.getenv('GOOGLE_API_KEY'):
    raise ValueError('Add GOOGLE_API_KEY to a .env file before running.')

In [4]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
);

In [5]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explain:str

In [6]:
def Generate_joke(state: JokeState):
    prompt = f"Write a short, one line joke on the topic :{state['topic']}"
    response = llm.invoke(prompt)
    state['joke'] = response.content


    return {
       "joke": state['joke']
    }


In [7]:
def Explain_joke(state: JokeState):
    prompt = f"Explain this joke {state['joke']} in 2-3 sentence in easy words."
    response = llm.invoke(prompt)
    state['explain'] = response.content

    return {
       "explain": state['explain']
    }


In [8]:
Graph = StateGraph(JokeState)

Graph.add_node('Generate_joke',Generate_joke)
Graph.add_node('Explain_joke',Explain_joke)

Graph.add_edge(START,'Generate_joke')
Graph.add_edge('Generate_joke','Explain_joke')
Graph.add_edge('Explain_joke',END)

Workflow = Graph.compile()

In [9]:
intial_state = {
    'topic' : 'python',
    'joke' : '',
    'explain' : ''
}

result = Workflow.invoke(intial_state)

print(result['joke'])
print(result['explain'])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': "My favorite part about Python is when it throws an IndentationError just to remind me who's really in control.", 'extras': {'signature': 'EmAKXgFpFH0TcJ+cjgnIkdNWLcy0KPb8JS09m+ueMf9s/80PITdqjk/mfDIcKi9g3O8+8vRiHr+DdULeMGjM2CnaRkkebhpEIp4MqLqQ+f1WEzHCH1mA7L/dWh4byCYo5jg='}}]
[{'type': 'text', 'text': 'This joke is about the programming language Python, which requires spaces (indentation) at the beginning of lines to organize the code. If you make even a tiny mistake with these spaces, Python crashes and gives you an "IndentationError." The joke is that Python acts like a strict boss, using these annoying errors to show the programmer who is really in charge.', 'extras': {'signature': 'EmAKXgFpFH0TTmcZAl0k4znbhFyb2p5cclCrJW0gQujNTX0SUKlK1OOmiG5FJICgxCCjnSGi8qrpLQsf2WEtJuS0vh0BVHYcjHatwwSxtzgnZTRW82rU5jA/TD2LgRBXk7g='}}]


In [21]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class JokeState(TypedDict):
    topic: str
    joke: Annotated[list[str], add]
    explain: Annotated[list[str], add]

def generate_joke(state: JokeState):
   
    prompt = f"Write a short, funny joke about {state['topic']}"
    response = llm.invoke(prompt)
    return {"joke": [response.content]}

def generate_explanation(state: JokeState):
    joke = state["joke"][-1]
    prompt = f"Explain this joke simply in 1-2 sentences: {joke}"
    response = llm.invoke(prompt)
    return {"explain": [response.content]}

Graph = StateGraph(JokeState)

Graph.add_node("generate_joke", generate_joke)
Graph.add_node("generate_explanation", generate_explanation)

Graph.add_edge(START, "generate_joke")
Graph.add_edge("generate_joke", "generate_explanation")
Graph.add_edge("generate_explanation", END)

checkpointer = InMemorySaver()
bot = Graph.compile(checkpointer=checkpointer)

config1 = {"configurable": {"thread_id": "1"}}

res1 = bot.invoke({"topic": "football"}, config=config1)

print("Thread 1 - Football")
print("Joke:", res1["joke"][-1])
print("Explanation:", res1["explain"][-1])

config2 = {"configurable": {"thread_id": "2"}}

res2 = bot.invoke({"topic": "python programming"}, config=config2)

print("\nThread 2 - Python")
print("Joke:", res2["joke"][-1])
print("Explanation:", res2["explain"][-1])

res3 = bot.invoke({"topic": "cricket"}, config=config1)

print("\nThread 1 - After Second Run")
print("All Topics:", res3["topic"])
print("All Jokes:", res3["joke"])
print("All Explanations:", res3["explain"])

Thread 1 - Football
Joke: [{'type': 'text', 'text': 'My wife told me to stop making football puns, but I said, "Come on, referee, that’s a terrible call!" \n\nShe looked at me and said, "That’s it, you’re benched for the rest of the season."', 'extras': {'signature': 'EmAKXgFpFH0TWisPJXiBUu9HVYIuU9uac4H9ZioVsweDxS+3TKbhvQeBRwjRw/ZYuinqDOgczds9XXpKjLkplZc/kVvlpJ1ajo3L0GPSKY6QVRd21qiX9FO9Gvk2A+cUL/U='}}]
Explanation: [{'type': 'text', 'text': 'The husband pretends his wife is a football referee making a bad decision by telling him to stop making football puns. In response, she uses a football term herself by "benching" him for the rest of the season.', 'extras': {'signature': 'EmAKXgFpFH0TvxBXq1SxxDwaBmCVnCXj7zuGC3h9W3dwNDt105ogIrSNnjNuhkXvyt/Jw/OV+kjfwHU/im3oGF6lGrd17U8VUsXnqZ9iUtByk/Kz2QEA3N0WTPajHKO5ggo='}}]

Thread 2 - Python
Joke: [{'type': 'text', 'text': "Why do Python programmers prefer wearing glasses?\n\nBecause they don't C#!", 'extras': {'signature': 'EmAKXgFpFH0TzYLnCPY/ZHPT

In [20]:
snapshot1 = bot.get_state(config1)
snapshot2 = bot.get_state(config2)

print("Thread 1 Saved State:", snapshot1.values)
print("Thread 2 Saved State:", snapshot2.values)

Thread 1 Saved State: {'topic': ['football', 'cricket'], 'joke': [[{'type': 'text', 'text': 'My wife told me to stop making football pun jokes around the house.\n\nI said, "Okay, point taken... but that’s a foul!" \n\nShe looked at me and said, "Go to the penalty box."', 'extras': {'signature': 'EmAKXgFpFH0TASqmX9Vt1U/9mpLgZzfRz1xEgyNyJCtaAkd33Llu+oyYprMbBayBTntKznYvDD/MMRpyIEPkweiWQBoHqUQAu/cyIjUN3FrbN+s9zRq0QTv+OF1+TJQzbfk='}}], [{'type': 'text', 'text': 'Why do cricket players love to drink tea?\n\nBecause every time they go out to the crease, they know they’ll be having a break in two minutes!', 'extras': {'signature': 'EmAKXgFpFH0T4myI6L4EjBTxS5D1y1u6MrYw/bLptrvrhJHGTgwTmVywxKlGpSSg7ApA91mYd/1xNmjFVdve3x32h0uOYMkkJRGb0iz/r/zAWmsSh3oYHW1Y9Ei5ZKI/ibA='}}]], 'explain': [[{'type': 'text', 'text': 'The husband agrees to stop making football jokes, but immediately uses two more football terms ("point taken" and "foul") in his response. His wife then punishes him by telling him to go to 